In [ ]:
!pip install -q -U \
  "transformers>=4.53.0" \
  accelerate \
  timm \
  librosa \
  soundfile \
  pandas \
  scikit-learn \
  tqdm \
  huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 42.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which i

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test_dataset_final.csv to test_dataset_final.csv


In [ ]:
import zipfile
import os
import glob

zip_files = glob.glob("audio_files*.zip")
print("Found zip files:", zip_files)

zip_path = zip_files[0]
audio_dir = "audio_files"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(".")

print("Files extracted.")
print("Checking audio folder:")
print(os.listdir(audio_dir)[:10])
print("Total audio files:", len(os.listdir(audio_dir)))

Found zip files: ['audio_files.zip']
Files extracted.
Checking audio folder:
['0_1_d1680.wav', '12_0_d66.wav', '0_0_d749.wav', '1_1_d2027.wav', '1_0_d706.wav', '2_0_d1891.wav', '1_0_d762.wav', '0_1_d1817.wav', '1_0_d2539.wav', '12_1_d1292.wav']
Total audio files: 404


In [1]:
import os
import re
import pandas as pd
import torch
import librosa

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoProcessor, Gemma3nForConditionalGeneration

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Audio column:", audio_col)
print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

missing = []

for f in df[audio_col].astype(str).tolist():
    p = os.path.join(audio_dir, f)
    if not os.path.exists(p):
        missing.append(f)

print("Missing audio files:", len(missing))

if len(missing) > 0:
    print("First missing files:", missing[:10])
    raise FileNotFoundError("Some audio files from CSV are not present in audio_files folder.")

print("\nChecking first 3 audio-text pairs:")

for i in range(min(3, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    label = str(df.loc[i, label_col])

    print("\nSample", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)
    print("True label:", label)

model_name = "google/gemma-3n-E2B-it"

print("\nLoading model:", model_name)

use_gpu = torch.cuda.is_available()

if not use_gpu:
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

if torch.cuda.is_bf16_supported():
    dtype = torch.bfloat16
    print("Using dtype: bfloat16")
else:
    dtype = torch.float32
    print("Using dtype: float32 because this GPU may not support bfloat16")

processor = AutoProcessor.from_pretrained(model_name)

model = Gemma3nForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()

print("Model loaded successfully.")

def make_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def clean_pred(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

def load_audio(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = audio.astype("float32")
    return audio, sr

def get_first_model_device():
    return next(model.parameters()).device

def move_inputs_to_model_device(inputs):
    dev = get_first_model_device()

    moved = {}

    for k, v in inputs.items():
        if torch.is_tensor(v):
            if v.dtype.is_floating_point:
                moved[k] = v.to(dev, dtype=dtype)
            else:
                moved[k] = v.to(dev)
        else:
            moved[k] = v

    return moved

def print_input_debug(i, audio_path, transcript, audio, sr, messages, inputs):
    print("\n" + "=" * 80)
    print("DEBUG SAMPLE:", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Loaded audio sampling rate:", sr)
    print("Loaded audio shape:", audio.shape)
    print("Loaded audio dtype:", audio.dtype)
    print("Transcript:", transcript)

    print("\nMessage content types:")
    for msg in messages:
        print("Role:", msg["role"])
        for item in msg["content"]:
            print("  type:", item["type"])

    print("\nProcessor input keys:")
    print(list(inputs.keys()))

    print("\nProcessor input tensor details:")
    for k, v in inputs.items():
        if torch.is_tensor(v):
            print(k, "shape:", tuple(v.shape), "dtype:", v.dtype, "device before move:", v.device)
        else:
            print(k, "type:", type(v))

    if "input_ids" in inputs:
        print("Text input token length:", inputs["input_ids"].shape[-1])

    if "input_features" in inputs:
        print("Audio input_features shape:", tuple(inputs["input_features"].shape))

    if "input_features_mask" in inputs:
        print("Audio input_features_mask shape:", tuple(inputs["input_features_mask"].shape))

    print("=" * 80 + "\n")

def predict(audio_path, transcript, i=None, debug=False):
    prompt = make_prompt(transcript)

    audio, sr = load_audio(audio_path)

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are a helpful assistant that classifies sentence type from speech audio and transcript."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio": audio
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    if debug:
        print_input_debug(i, audio_path, transcript, audio, sr, messages, inputs)

    input_len = inputs["input_ids"].shape[-1]

    inputs = move_inputs_to_model_device(inputs)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    new_tokens = outputs[0][input_len:]

    new_text = processor.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    full_text = processor.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if debug:
        print("\nRaw decoded FULL text:")
        print(full_text)

        print("\nRaw decoded NEW text only:")
        print(new_text)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    pred = clean_pred(ans)

    if debug:
        print("Cleaned prediction:", pred)
        print("-" * 80)

    return pred

print("\nRunning debug predictions on first 5 samples only:")

debug_preds = []

for i in range(min(5, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    true_label = str(df.loc[i, label_col])

    pred = predict(audio_path, transcript, i=i, debug=True)

    debug_preds.append(pred)

    print("Sample:", i)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

print("\nNow running on full dataset...")

preds = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])

    pred = predict(audio_path, transcript, i=i, debug=False)
    preds.append(pred)

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[audio_col, text_col, label_col, "predicted_type"]].head(20))

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("gemma3n_speech_text_predictions.csv", index=False)

print("Saved predictions to gemma3n_speech_text_predictions.csv")

from google.colab import files
files.download("gemma3n_speech_text_predictions.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='str')
Audio column: file_name
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']
Missing audio files: 0

Checking first 3 audio-text pairs:

Sample 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Transcript: Hello, Jane. I'm really glad to meet you here.
True label: declarative

Sample 1
Audio 

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1526 [00:00<?, ?it/s]

Model loaded successfully.

Running debug predictions on first 5 samples only:

DEBUG SAMPLE: 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Loaded audio sampling rate: 16000
Loaded audio shape: (44000,)
Loaded audio dtype: float32
Transcript: Hello, Jane. I'm really glad to meet you here.

Message content types:
Role: system
  type: text
Role: user
  type: audio
  type: text

Processor input keys:
['input_ids', 'attention_mask', 'token_type_ids', 'input_features', 'input_features_mask']

Processor input tensor details:
input_ids shape: (1, 493) dtype: torch.int64 device before move: cpu
attention_mask shape: (1, 493) dtype: torch.int64 device before move: cpu
token_type_ids shape: (1, 493) dtype: torch.int64 device before move: cpu
input_features shape: (1, 272, 128) dtype: torch.float32 device before move: cpu
input_features_mask shape: (1, 272) dtype: torch.bool device before move: cpu
Text input token length: 493
Audio input_features shape: (1, 27

100%|██████████| 404/404 [09:56<00:00,  1.48s/it]


Prediction counts:
predicted_type
exclamatory      121
declarative      116
interrogative    107
imperative        60
Name: count, dtype: int64

First 20 predictions:
        file_name                                         transcript  \
0     0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1     0_0_d47.wav         I heard you and James are engaged at last.   
2    0_0_d215.wav                            Give me scotch, please.   
3     0_0_d49.wav            Hi, what brings you to my office today?   
4    0_0_d159.wav  Please help yourself at your dishes. I hope yo...   
5     0_0_d50.wav                                  Hello, Jack here.   
6    0_0_d403.wav            Excuse me. I wonder if you can help me.   
7     0_0_d63.wav            Umm, Jenny, are you having a good time?   
8    0_0_d681.wav  Excuse me. I bought this shirt yesterday, but ...   
9    0_0_d684.wav     Watch out! You're too close to the fire place.   
10   0_0_d704.wav                       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>